# 关于模型调用的invoke的使用

## 1.invoke的传参

### 1.1文本输入

举例

In [19]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model


#读取.env配置文件中的信息。
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

#模型的初始化
model = init_chat_model(
    api_base=DEEPSEEK_BASE_URL,
    base_url=DEEPSEEK_BASE_URL,
    model='deepseek:deepseek-v4-flash',
)

In [20]:
response = model.invoke("翻译如下文字：你好")
print(response.content)

Hello.


## 1.2 字典列表

举例1

In [21]:
messages = [
    {"role":"system","content":"你是一个专业的数学老师。"},
    {"role":"user","content":"什么是斐波那契数列？"}
]
response = model.invoke(messages)
print(response.content)

斐波那契数列是一个由“前两项相加得到下一项”的递推数列。

最常见的定义是：

\[
F_0=0,\quad F_1=1,\quad F_n=F_{n-1}+F_{n-2}\quad(n\ge 2)
\]

于是得到：

\[
0,1,1,2,3,5,8,13,21,34,55,89,\dots
\]

也有另一种常见写法是从 \(F_1=F_2=1\) 开始：

\[
1,1,2,3,5,8,13,21,34,55,89,\dots
\]

核心规则就是：**从第三项开始，每一项都等于前两项之和**。例如：

\[
2=1+1,\quad 3=1+2,\quad 5=2+3,\quad 8=3+5
\]

斐波那契数列有一个很著名的通项公式，叫比内公式：

\[
F_n=\frac{\varphi^n-\psi^n}{\sqrt5}
\]

其中

\[
\varphi=\frac{1+\sqrt5}{2}\approx 1.618,\quad
\psi=\frac{1-\sqrt5}{2}
\]

\(\varphi\) 就是黄金比例。虽然斐波那契数列每一项都是整数，但它的通项公式却用到了无理数。

它还有一些重要性质：

- 相邻两项的比值越来越接近黄金比例：
  \[
  \lim_{n\to\infty}\frac{F_{n+1}}{F_n}=\varphi
  \]

- 前 \(n\) 项和：
  \[
  F_1+F_2+\cdots+F_n=F_{n+2}-1
  \]

- 卡西尼恒等式：
  \[
  F_{n+1}F_{n-1}-F_n^2=(-1)^n
  \]

斐波那契数列最早来自“兔子繁殖问题”：假设一对兔子每月生一对新兔子，新兔子出生后第二个月开始繁殖，且兔子不死，那么每个月的兔子对数就构成斐波那契数列。

它在数学、计算机算法、黄金分割、植物叶序、松果和向日葵种子排列等现象中都经常出现。不过要注意，自然界中并不是“到处都严格是斐波那契数列”，更多是某些生长模式会接近它。

简单说：**斐波那契数列是一个从 0 和 1 开始，每一项等于前两项之和的著名数列。**


In [22]:
messages = [
    {"role":"system","content":"你是一个专业的数学老师。"},
    {"role":"user","content":"2 + 3 * 2 = ？"},
    {"role":"assistant","content":"8"},
    {"role":"user","content":"我刚才问了什么问题？"}
]

response = model.invoke(messages)
# 打印响应
print(f"AI的回复:{response.content}")

AI的回复:你刚才问的是：

**2 + 3 * 2 = ？**


传入记忆

In [25]:
conversation = [
    {"role": "system", "content": "你是一个非常友好的AI助手"},
    {"role": "user", "content": "你好，我叫小明"},
]
# 第一次对话
response1 = model.invoke(conversation)
# 打印响应
print(f"AI的回复1:{response1.content}")


conversation.append({'role':'assistant','content':response1.content})
conversation.append({'role':'user','content':'我叫什么名字'})
# 第二次对话
response2 = model.invoke(conversation)
# 打印响应
print(response2.content)

AI的回复1:你好，小明！很高兴认识你 😊  
我是你的AI助手，有什么我可以帮你的吗？
你叫小明呀！😊 有什么我可以帮你的吗？


## 1.3消息对象列表

举例1

In [26]:
from langchain_core.messages import SystemMessage, AIMessage, HumanMessage

messages = [
    SystemMessage("你是一个专业的数学老师。"),
    HumanMessage("2 + 3 * 2 = ?"),
    AIMessage("8"),
    HumanMessage("我刚才问什么问题了？")
]

response = model.invoke(messages)
print(f'{response.content}')

你刚才问的是：**“2 + 3 * 2 = ?”**

答案是 **8**。


举例2：传递记忆

In [31]:
from langchain_core.messages import SystemMessage, AIMessage, HumanMessage

messages = [
    SystemMessage("你是一个专业的数学老师。"),
    HumanMessage("2 + 3 * 2 = ?"),
    AIMessage("8"),
    HumanMessage("我刚才问什么问题了？")
]
#第一次对话
response = model.invoke(messages)
print(f'{response.content}')

#添加记忆
messages.append(AIMessage(content=response.content))
messages.append(HumanMessage(content='我刚才问了什么问题'))

#第二次对话
response2 = model.invoke(messages)
print(response2.content)

你刚才问的是：**“2 + 3 * 2 = ?”**
你刚才问的是：**“我刚才问什么问题了？”**

再往前一次，你问的是：**“2 + 3 * 2 = ?”**


## 2.invoke返回值

举例1


In [32]:
response = model.invoke([HumanMessage(content='2+3*2 = ?')])

In [33]:
print(type(response))

<class 'langchain_core.messages.ai.AIMessage'>


In [ ]:
print(response)

content='8' additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer simple arithmetic. Need obey precedence multiplication before addition: 3*2=6 +2=8. Final concise.'} response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 37, 'total_tokens': 66, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 27, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 37}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '7e5952e5-3ef5-4afc-94d6-1a038d417e2f', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a120bb-dafc-7f63-9b27-06538a316473-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 37, 'output_tokens': 29, 'total_tokens': 66, 'input_token_details': {'cache_read': 0}, 'output_token_de

美化输出

In [35]:
from rich import print as rprint

rprint(response)

AIMessage(
    content='8',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 'We need answer simple arithmetic. Need obey precedence multiplication before 
addition: 3*2=6 +2=8. Final concise.'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 29,
            'prompt_tokens': 37,
            'total_tokens': 66,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 27,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 37
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
        'id': '7e5952e5-3ef5-4afc-94d6-1a038d417e2f',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a120bb-dafc-7f63-9b27-06538a316473-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 37,
        'output_tokens': 29,
        'total_tokens': 66,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 27}
    }
)

举例2：

In [36]:
response = model.invoke("用一句话解释什么是 AI")

# 1. 获取回复内容
print("AI 回复:", response.content)

# 2. 获取响应元数据
metadata = response.response_metadata
print(f"使用的模型: {metadata['model_name']}")
print(f"结束原因: {metadata['finish_reason']}")
print(f"模型提供商：{metadata['model_provider']}\n")

# 3. 获取 Token 使用情况
usage = metadata.get('token_usage', {})
print(f"输入 tokens: {usage.get('prompt_tokens')}")
print(f"输出 tokens: {usage.get('completion_tokens')}")
print(f"总计 tokens: {usage.get('total_tokens')}")

# 4. 获取消息 ID
print(f"消息 ID: {response.id}")

AI 回复: AI（人工智能）是让机器模拟人类智能（如学习、推理、感知和决策）来执行通常需要人类智能的任务的技术。
使用的模型: deepseek-flash
结束原因: stop
模型提供商：deepseek

输入 tokens: 35
输出 tokens: 120
总计 tokens: 155
消息 ID: lc_run--01a120c8-d077-7431-90b5-e392c658d8d7-0
